# Task A Grad-CAM 実行 Notebook（AI-agent primary model）

凍結済みの primary model に対する **Grad-CAM 生成専用**の Notebook です。上から順に実行すれば完了します。

## 固定条件（この Notebook では変更しない）

| 項目 | 値 |
|---|---|
| primary model | `lr3e-4_aug_b` / **seed 42** |
| 閾値 | `final_selection.json` に凍結された Youden 値（**Test で再計算しない**） |
| 対象層 | ResNet18 `layer4[-1]` |
| 症例 | TP 4・FP 4・TN 4・**FN 3（Test に 3 例しか存在しない）= 計 15 例**。他群から補充しない |
| Test の扱い | 既存の Test 予測を読むだけ。**再学習・再選択・再評価はしない** |
| 参照範囲 | 「気合のCOVID19」配下のみ。「旧」で始まるファイル、「心機一転COVID19」は参照しない |

出力先に既存ファイルがある場合、スクリプトは停止します（置き換える場合のみ Cell 8 の `ALLOW_OVERWRITE = True`）。

## Cell 1: Google Drive を mount する

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: パス定義

入力はすべて「気合のCOVID19」配下。書き込みは `OUT_DIR` と `/content` のみです。

In [ ]:
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/気合のCOVID19')
PROJECT_ROOT = DRIVE_ROOT / '01_TaskA_CXR'

# コード一式（ZIP）。別の版を使う場合はファイル名だけ変更する。
ZIP_NAME        = 'AIagent_code_20260921h.zip'
ZIP_PATH        = DRIVE_ROOT / '00_共通・研究管理' / 'AIagent_code' / ZIP_NAME
# ZIP と同じ場所に置かれた *_manifest.json から期待 SHA256 を読む。
# manifest が無い場合は下の固定値を使う（AIagent_code_20260921h.zip の hash）。
EXPECTED_SHA256_FALLBACK = '9e3c65d61e12a187c2840e4dc98c125fa196187e565a68da47159a645cae98ce'
EXTRACT_DIR = Path('/content/COVID19_AI_Agent')

RUNS_ROOT             = PROJECT_ROOT / '04_Training' / 'AIagent_taskA_runs'
FINAL_SELECTION_PATH  = RUNS_ROOT / 'final_selection.json'
TEST_PREDICTIONS_PATH = PROJECT_ROOT / '05_Evaluation' / 'AIagent_taskA_test' / 'test_predictions_primary.csv'
IMAGE_DIR             = PROJECT_ROOT / '01_前処理' / 'AIagent_taskA_png512_16bit' / 'images'
OUT_DIR               = PROJECT_ROOT / '06_GradCAM' / 'AIagent_taskA_primary_test'

for name, p in [('ZIP_PATH', ZIP_PATH), ('RUNS_ROOT', RUNS_ROOT),
                ('FINAL_SELECTION_PATH', FINAL_SELECTION_PATH),
                ('TEST_PREDICTIONS_PATH', TEST_PREDICTIONS_PATH),
                ('IMAGE_DIR', IMAGE_DIR), ('OUT_DIR', OUT_DIR)]:
    print(f'{name:22s} {p}')

## Cell 3: ZIP の存在確認と SHA256 検証

配布時と同じコードであることを確認します。一致しなければここで停止します。

In [ ]:
import hashlib, json

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

if not ZIP_PATH.exists():
    raise FileNotFoundError(f'ZIP が見つかりません: {ZIP_PATH}\n'
                            f'Drive の 00_共通・研究管理/AIagent_code/ に配置してください。')

manifest_path = ZIP_PATH.with_name(ZIP_PATH.stem + '_manifest.json')
if manifest_path.exists():
    expected = json.loads(manifest_path.read_text(encoding='utf-8'))['sha256']
    source = f'manifest ({manifest_path.name})'
else:
    expected = EXPECTED_SHA256_FALLBACK
    source = 'notebook 内の固定値'

actual = sha256(ZIP_PATH)
print(f'zip      : {ZIP_PATH.name}')
print(f'actual   : {actual}')
print(f'expected : {expected}  ({source})')
if actual != expected:
    raise RuntimeError('ZIP の SHA256 が期待値と一致しません。配置した ZIP の版を確認してください。'
                       '別の版を使う場合は Cell 2 の ZIP_NAME と EXPECTED_SHA256_FALLBACK を更新します。')
print('\nOK: ZIP は配布時と同一です')

## Cell 4: コードの展開

`/content/COVID19_AI_Agent` に展開します。Drive 上の ZIP は読むだけで変更しません。
前回の残骸が混ざらないよう、展開先が既にあれば削除してから展開します（削除対象は `/content` 配下のみ）。

In [ ]:
import shutil, zipfile, os

assert str(EXTRACT_DIR).startswith('/content/'), '展開先は /content 配下に限定します'
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH) as z:
    bad = [n for n in (i.orig_filename for i in z.infolist()) if '\\' in n]
    if bad:
        raise RuntimeError(f'ZIP 内の entry 名にバックスラッシュがあります（Linux で展開できません）: {bad[:3]}')
    z.extractall(EXTRACT_DIR)

script = EXTRACT_DIR / 'scripts/18_taskA_gradcam_primary_test.py'
required = [script,
            EXTRACT_DIR / 'src/covid_mortality/evaluation/gradcam.py',
            EXTRACT_DIR / 'src/covid_mortality/models/taskA_resnet18.py',
            EXTRACT_DIR / 'src/covid_mortality/data/taskA_dataset.py',
            EXTRACT_DIR / 'data/splits/COVID19_固定患者split_1277.csv',
            EXTRACT_DIR / 'data/interim/cxr_audit/index_cxr_manifest_window_T0m2_T0.csv']
missing = [str(p.relative_to(EXTRACT_DIR)) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f'展開後に見つかりません: {missing}')

os.chdir(EXTRACT_DIR)
print('展開先 :', EXTRACT_DIR)
print('ファイル数 :', sum(1 for _ in EXTRACT_DIR.rglob('*') if _.is_file()))
print('実行スクリプト :', script.relative_to(EXTRACT_DIR))
print('\nOK: 必要なファイルがそろっています')

## Cell 5: 実行環境の確認

Colab には torch / torchvision / numpy / pandas / Pillow が既定で入っています。
**足りないものだけ**を入れ、既にあるものは再インストールしません。
展開したコードに `requirements.txt` / `pyproject.toml` があればそちらを優先します。

In [ ]:
import importlib, subprocess, sys

req_file = next((p for p in [EXTRACT_DIR / 'requirements.txt', EXTRACT_DIR / 'pyproject.toml',
                             EXTRACT_DIR / 'uv.lock'] if p.exists()), None)
if req_file is not None and req_file.name == 'requirements.txt':
    print(f'{req_file.name} を使用します')
    print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)],
                         capture_output=True, text=True).stdout[-500:])
elif req_file is not None:
    print(f'{req_file.name} を検出しました（Colab では個別インストールは行いません）')

needed = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy',
          'pandas': 'pandas', 'PIL': 'Pillow'}   # Grad-CAM に pydicom は不要
missing = []
for module, package in needed.items():
    try:
        m = importlib.import_module(module)
        print(f'{module:12s} {getattr(m, "__version__", "ok")}')
    except ImportError:
        missing.append(package)

if missing:
    print('不足しているパッケージを導入します:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)

import torch
print('\ncuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU で実行します（Grad-CAM は 15 例なので CPU でも可）')

## Cell 6: 入力ファイルの存在確認

1 つでも欠けていれば、Grad-CAM を実行せずにここで停止します。

In [ ]:
import pandas as pd

problems = []
for label, p in [('final_selection.json', FINAL_SELECTION_PATH),
                 ('test_predictions_primary.csv', TEST_PREDICTIONS_PATH),
                 ('image directory', IMAGE_DIR)]:
    ok = p.exists()
    print(f'{label:32s} {"OK" if ok else "見つかりません"}  {p}')
    if not ok:
        problems.append(label)

if not problems:
    frozen = json.loads(FINAL_SELECTION_PATH.read_text(encoding='utf-8'))
    ckpt = Path(frozen['primary_analysis']['checkpoint'])
    print(f'{"primary checkpoint":32s} {"OK" if ckpt.exists() else "見つかりません"}  {ckpt}')
    if not ckpt.exists():
        problems.append('primary checkpoint')

    n_images = len(list(IMAGE_DIR.glob('*.png')))
    print(f'\n画像枚数 : {n_images}（期待 1277）')
    if n_images != 1277:
        problems.append(f'画像枚数が 1277 ではありません（{n_images}）')

    pred = pd.read_csv(TEST_PREDICTIONS_PATH, encoding='utf-8-sig')
    print(f'Test 予測 : {len(pred)} 行、死亡 {int(pred.true_label.sum())} 例')
    if len(pred) != 128:
        problems.append(f'Test 予測が 128 行ではありません（{len(pred)}）')

if problems:
    raise RuntimeError(f'入力が不足しています: {problems}\nGrad-CAM は実行しません。')
print('\nOK: 入力はそろっています')

## Cell 7: 凍結済み primary model の確認

`lr3e-4_aug_b` / seed 42 でなければ停止します。閾値は表示するだけで、再計算しません。

In [ ]:
primary = frozen['primary_analysis']
threshold = primary['threshold']['value']

print('primary model      :', primary['model'])
print('checkpoint         :', primary['checkpoint'])
print('checkpoint sha256  :', primary['checkpoint_sha256'][:32], '…')
print('Youden threshold   :', threshold)
print('threshold rule     :', primary['threshold'].get('rule'))
print('tie break          :', primary['threshold'].get('tie_break'))
print('Test で再計算      :', primary['threshold'].get('recomputed_on_test'))

if primary['model'] != 'lr3e-4_aug_b/seed42':
    raise RuntimeError(f"primary model が lr3e-4_aug_b/seed42 ではありません: {primary['model']}\n"
                       f'別の seed や ensemble を主解析にすることはできません。')
if threshold is None:
    raise RuntimeError('Youden threshold を取得できません。')
print('\nOK: 固定条件と一致しています')

## Cell 8: Grad-CAM の実行

`scripts/18_taskA_gradcam_primary_test.py` を実行します。スクリプト側でも
（primary model の一致、checkpoint の hash、Test 予測 128 例、群ごとの症例数、画像の存在、出力先が空か）を
再確認し、満たさなければ何も書かずに停止します。

In [ ]:
ALLOW_OVERWRITE = False   # 既存の成果物を置き換える場合のみ True にする

cmd = [sys.executable, 'scripts/18_taskA_gradcam_primary_test.py', '--project', '.',
       '--frozen', str(FINAL_SELECTION_PATH),
       '--test-predictions', str(TEST_PREDICTIONS_PATH),
       '--image-dir', str(IMAGE_DIR),
       '--out-dir', str(OUT_DIR)]
if ALLOW_OVERWRITE:
    cmd.append('--allow-overwrite')

print(' '.join(cmd), '\n')
result = subprocess.run(cmd, cwd=EXTRACT_DIR, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('--- stderr ---')
    print(result.stderr)
    raise RuntimeError(f'Grad-CAM の実行に失敗しました（return code {result.returncode}）。'
                       f'上のメッセージで、どこまで進んで何で止まったかを確認してください。')
print('SUCCESS: Grad-CAM の生成が完了しました')

## Cell 9: 出力の確認

成果物がそろっているか、症例数が 15（TP 4 / FP 4 / TN 4 / FN 3）かを確認します。

In [ ]:
expected_files = ['gradcam_selection.csv', 'gradcam_notes.md', 'gradcam_run_meta.json',
                  'panels/gradcam_panel_4x4.png']
missing = [f for f in expected_files if not (OUT_DIR / f).exists()]
individual = sorted((OUT_DIR / 'individual').glob('*.png'))
print('出力先 :', OUT_DIR)
for f in expected_files:
    print(f'  {"OK" if (OUT_DIR / f).exists() else "なし":4s} {f}')
print(f'  {"OK" if individual else "なし":4s} individual/ ({len(individual)} 枚)')
if missing or not individual:
    raise RuntimeError(f'出力が不足しています: {missing or "individual images"}')

sel = pd.read_csv(OUT_DIR / 'gradcam_selection.csv', encoding='utf-8-sig')
counts = sel.outcome_group.value_counts().to_dict()
print(f'\n症例数 : {len(sel)}（期待 15）')
print('群構成 :', {g: counts.get(g, 0) for g in ('TP', 'FP', 'TN', 'FN')})
if len(sel) != 15 or {g: counts.get(g, 0) for g in ('TP', 'FP', 'TN', 'FN')} != {'TP': 4, 'FP': 4, 'TN': 4, 'FN': 3}:
    raise RuntimeError('症例数または群構成が想定（15 例 / TP4・FP4・TN4・FN3）と異なります。')

display(sel[['subject_id', 'outcome_group', 'selection_rank_within_group', 'true_label',
             'predicted_probability', 'predicted_label', 'frac_in_padding',
             'frac_in_border_band', 'frac_in_central_50']])
print('\nOK: 15 例（TP 4 / FP 4 / TN 4 / FN 3）')

## Cell 10: 実行メタ情報の確認

In [ ]:
meta = json.loads((OUT_DIR / 'gradcam_run_meta.json').read_text(encoding='utf-8'))
print('selection_note            :', meta.get('selection_note') or '(なし)')
print('cases_available_per_group :', meta.get('cases_available_per_group'))
print('cases_used_per_group      :', meta.get('cases_used_per_group'))
print('panel_empty_cells         :', meta.get('panel_empty_cells'))
print('gradcam_layer             :', meta.get('gradcam_layer'))
print('threshold_used            :', meta.get('threshold_used'),
      '(Test で再計算:', meta.get('threshold_recomputed_on_test'), ')')
print('warnings                  :', meta.get('warnings') or '(なし)')
print('\n出力ファイル数 :', len(meta.get('outputs', [])))

## Cell 11: 4×4 パネルの表示

FN が 3 例のため、右下の 1 セルは空欄です（他群から補充していません）。

In [ ]:
from IPython.display import Image as IPyImage, display as ipy_display

panel = OUT_DIR / 'panels' / 'gradcam_panel_4x4.png'
print(panel)
ipy_display(IPyImage(filename=str(panel), width=1100))

## Cell 12: 最終サマリー

In [ ]:
print('=' * 72)
print('Task A Grad-CAM 生成 完了')
print('=' * 72)
print(f'primary model   : {meta["model"]}（condition {meta["condition"]} / seed {meta["seed"]}）')
print(f'threshold       : {meta["threshold_used"]}（{meta["threshold_source"]}、Test で再計算なし）')
print(f'checkpoint      : {meta["checkpoint"]}')
print(f'                  sha256 {meta["checkpoint_sha256"][:32]}…')
print(f'Grad-CAM 層     : {meta["gradcam_layer"]}')
print(f'出力先          : {OUT_DIR}')
print(f'症例構成        : 計 {len(sel)} 例  '
      f'TP {counts.get("TP", 0)} / FP {counts.get("FP", 0)} / TN {counts.get("TN", 0)} / FN {counts.get("FN", 0)}')
if meta.get('selection_note'):
    print(f'                  {meta["selection_note"]}')
print(f'個別画像        : {len(individual)} 枚')
print(f'パネル          : panels/gradcam_panel_4x4.png（空セル {meta.get("panel_empty_cells")}）')
print('Drive 保存      : 完了')
print('\n次の作業 : gradcam_notes.md の「総括（読影）」に、画像を確認したうえで所見を記入する')